# 🌍 Ecosystem Condition Mapping from FAO LCCS Level 4 Land Cover

This notebook provides a complete workflow to derive an **Ecosystem Condition Map** from a Land Cover map classified under the **FAO Land Cover Classification System (LCCS) Level 4**.

### Framework References
- **FAO LCCS v3** — Classification system
- **IUCN Global Ecosystem Typology (GET) v2.1** — Ecosystem type crosswalk
- **UN SEEA EA (2021)** — Condition scoring (5-point scale)
- **IUCN Red List of Ecosystems (RLE)** — Collapse risk assessment

### Workflow Overview
1. Setup & Data Loading
2. LCCS L4 → Ecosystem Type Crosswalk
3. Compute Biophysical Condition Indicators
4. Score Ecosystem Condition (SEEA EA)
5. Spatial Aggregation & Map Production
6. Accuracy Assessment & Statistics
7. Export Outputs

> **Note:** This notebook works with local raster/vector files. For cloud-scale processing, see the Google Earth Engine appendix at the end.

---
## 0. Install & Import Dependencies

In [ ]:
# Install required packages (run once)
# !pip install rasterio geopandas numpy pandas matplotlib scipy scikit-learn rasterstats folium earthengine-api

In [ ]:
import os
import json
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
try:
  import geopandas as gpd
except:
  !pip install geopandas
  import geopandas as gpd
import rasterio
from rasterio.plot import show
from rasterio.transform import from_bounds
from rasterio.warp import calculate_default_transform, reproject, Resampling
from rasterio.mask import mask
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.patches as mpatches
from matplotlib.colors import ListedColormap, BoundaryNorm
from scipy import ndimage
try:
   import sklearn
except:
   !pip install scikit-learn

from sklearn.metrics import confusion_matrix, classification_report

# Optional: zonal statistics
try:
    from rasterstats import zonal_stats
    HAS_RASTERSTATS = True
except ImportError:
    HAS_RASTERSTATS = False
    print("rasterstats not available — install with: pip install rasterstats")
    !pip install rasterstats

print("✅ All core libraries loaded successfully")
print(f"   rasterio: {rasterio.__version__}")
print(f"   geopandas: {gpd.__version__}")
print(f"   numpy: {np.__version__}")

---
## 1. Configuration & File Paths

Set your input file paths and parameters here.

In [ ]:
# ============================================================
# USER CONFIGURATION — Edit these paths and parameters
# ============================================================

# Input files
LCCS_MAP_PATH      = "/Users/gregorygiuliani/Desktop/output/ecosystem_extent/LE_D26_basilicata_2025_maesL2.tif"        # FAO LCCS L4 land cover raster
NDVI_CURRENT_PATH  = "/Users/gregorygiuliani/Desktop/output/Basilicata/NDVI_current_Basilicata_2025_clipped.tif"        # Current-period NDVI (mean composite)
NDVI_REFERENCE_PATH = "/Users/gregorygiuliani/Desktop/output/Basilicata/NDVI_ref_Basilicata_Layer_clipped.tif"     # Reference-period NDVI (baseline)
CANOPY_COVER_PATH  = "/Users/gregorygiuliani/Desktop/output/Basilicata/Tree_Cover_Basilicata_2000_clipped.tif"        # Tree canopy cover % (Hansen GFC, native ~30m — resampled to LCCS 10m grid below)
#STUDY_AREA_PATH    = "/Users/gregorygiuliani/Library/CloudStorage/OneDrive-unige.ch/HE_LandShift/LivingSpaces/gpkg/basilicata.gpkg"         # Study area boundary (optional)

# Output directory
OUTPUT_DIR = "/Users/gregorygiuliani/Desktop/output/Basilicata/2025/"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Coordinate Reference System (EPSG code)
TARGET_CRS = "EPSG:4326"   # WGS84 — change to projected CRS for area calculations

# SEEA EA condition thresholds (% deviation from reference NDVI)
# Scores: 5=Very Good, 4=Good, 3=Fair, 2=Poor, 1=Very Poor
CONDITION_THRESHOLDS = {
    'very_good': 0.05,   # ≤5% deviation
    'good':      0.25,   # 5–25%
    'fair':      0.50,   # 25–50%
    'poor':      0.75,   # 50–75%
    # >75% = very poor/degraded
}

print("✅ Configuration set")
print(f"   Output directory: {os.path.abspath(OUTPUT_DIR)}")

---
## 2. LCCS Level 4 → Ecosystem Type Crosswalk

This crosswalk table maps FAO LCCS L4 codes to:
- **IUCN GET v2.1 biomes**
- **Ecosystem type names**
- **Ecosystem group** (for condition scoring strategy)

In [ ]:
# ============================================================
# LCCS L4 to Ecosystem Type Crosswalk Table
# Extend this table to match your study area's LCCS classes
# ============================================================
'''
crosswalk_data = [
    # LCCS_CODE, LCCS_L4_DESCRIPTION, IUCN_GET_BIOME, ECOSYSTEM_TYPE, ECO_GROUP
    # --- Forests ---
    (111, "Closed broadleaf evergreen forest",         "T1", "Tropical moist forest",            "forest"),
    (112, "Closed broadleaf deciduous forest",          "T2", "Tropical dry forest",               "forest"),
    (121, "Closed needle-leaved evergreen forest",      "T2", "Temperate conifer forest",          "forest"),
    (122, "Closed needle-leaved deciduous forest",      "T3", "Boreal/taiga forest",               "forest"),
    (131, "Open broadleaf evergreen forest",            "T1", "Tropical open forest",              "forest"),
    (132, "Open broadleaf deciduous forest",            "T4", "Tropical savanna woodland",         "forest"),
    (141, "Open needle-leaved evergreen forest",        "T2", "Dry sclerophyll woodland",          "forest"),
    (150, "Mixed forest (broadleaf + needle-leaved)",   "T2", "Temperate mixed forest",            "forest"),
    # --- Shrublands ---
    (211, "Closed broadleaf evergreen shrubland",       "T2", "Mediterranean shrubland (maquis)",  "shrubland"),
    (212, "Closed broadleaf deciduous shrubland",       "T4", "Subtropical shrubland",             "shrubland"),
    (220, "Open shrubland (sparse)",                    "T5", "Semi-arid shrubland",               "shrubland"),
    (230, "Needle-leaved shrubland",                    "T6", "Alpine/subalpine shrubland",        "shrubland"),
    # --- Grasslands & Savannas ---
    (311, "Closed herbaceous layer (grassland)",        "T4", "Tropical grassland",                "grassland"),
    (312, "Open herbaceous layer (grassland)",          "T5", "Temperate grassland / steppe",      "grassland"),
    (320, "Sparse herbaceous cover",                    "T5", "Dryland sparse grassland",          "grassland"),
    (330, "Mosaic herbaceous / shrubland",              "T4", "Savanna mosaic",                    "grassland"),
    # --- Wetlands ---
    (411, "Permanent herbaceous wetland",               "F2", "Freshwater marsh",                  "wetland"),
    (412, "Seasonal herbaceous wetland",                "F2", "Seasonal floodplain wetland",       "wetland"),
    (420, "Flooded forest",                             "TF1","Tropical flooded forest",           "wetland"),
    (430, "Mangrove",                                   "MFT1","Mangrove",                         "wetland"),
    # --- Croplands ---
    (511, "Irrigated cropland",                         None,  "Irrigated agriculture",            "cropland"),
    (512, "Rainfed cropland",                           None,  "Rainfed agriculture",              "cropland"),
    (520, "Mosaic cropland / natural vegetation",       None,  "Agricultural mosaic",              "cropland"),
    # --- Urban & Built-up ---
    (610, "Urban / built-up area",                      None,  "Urban/peri-urban",                 "urban"),
    (620, "Industrial area",                            None,  "Industrial",                       "urban"),
    # --- Bare / Sparse ---
    (710, "Bare rock / consolidated bare area",         "T7", "Rock outcrop",                      "bare"),
    (720, "Consolidated bare area (gravel/sand)",       "T7", "Desert/semi-desert",                "bare"),
    (730, "Sparse vegetation on bare substrate",        "T7", "Sparse dryland",                    "bare"),
    # --- Water & Ice ---
    (810, "Permanent water body",                       "F2", "Lake / reservoir",                  "water"),
    (820, "Temporary/seasonal water body",              "F2", "Seasonal lake / pond",              "water"),
    (910, "Snow and ice",                               "T6", "Glacier / permanent ice",           "ice"),
]
'''
crosswalk_data = [
    # LCCS_CODE, LCCS_L4_DESCRIPTION, IUCN_GET_BIOME, ECOSYSTEM_TYPE, ECO_GROUP
    (101, "Urban (Settlements and other artificial areas)", None, "Urban/peri-urban", "urban"),
    (102, "Cropland", None, "Agriculture", "cropland"),
    (103, "Grassland", "T5", "Temperate grassland", "grassland"),
    (104, "Forest and woodlands", "T2", "Temperate forest", "forest"),
    (105, "Heathland and shrub", "T2", "Mediterranean shrubland", "shrubland"),
    (106, "Sparsely vegetated land", "T7", "Sparse dryland", "bare"),
    (107, "Wetlands", "F2", "Mangrove", "wetland"),
    (200, "Rivers and lakes", "F2", "Lake / reservoir", "water"),
]
crosswalk_df = pd.DataFrame(
    crosswalk_data,
    columns=["lccs_code", "lccs_description", "iucn_get_biome", "ecosystem_type", "eco_group"]
)

# Create lookup dictionaries
code_to_type  = dict(zip(crosswalk_df.lccs_code, crosswalk_df.ecosystem_type))
code_to_group = dict(zip(crosswalk_df.lccs_code, crosswalk_df.eco_group))
code_to_biome = dict(zip(crosswalk_df.lccs_code, crosswalk_df.iucn_get_biome))

print(f"✅ Crosswalk table loaded: {len(crosswalk_df)} LCCS L4 classes mapped")
print(f"   Ecosystem groups: {crosswalk_df.eco_group.unique().tolist()}")
crosswalk_df.head(10)

---
## 3. Load & Explore Land Cover Data

In [ ]:
def load_raster(path, description=""):
    """Load raster and return array + metadata."""
    with rasterio.open(path) as src:
        array  = src.read(1).astype(np.float32)
        meta   = src.meta.copy()
        bounds = src.bounds
        crs    = src.crs
        nodata = src.nodata
    # Replace nodata with NaN
    if nodata is not None:
        array[array == nodata] = np.nan
    print(f"✅ Loaded {description or path}")
    print(f"   Shape: {array.shape} | CRS: {crs} | NoData: {nodata}")
    print(f"   Value range: [{np.nanmin(array):.3f}, {np.nanmax(array):.3f}]")
    return array, meta, bounds, crs


def align_raster(source_path, reference_path, resampling=Resampling.nearest, output_path=None):
    """
    Reproject and resample source raster to match reference raster
    (extent, resolution, CRS, and pixel dimensions).

    Parameters
    ----------
    source_path : str
        Path to the raster that needs to be resampled (e.g. Hansen tree cover, NDVI).
    reference_path : str
        Path to the raster whose grid (CRS, transform, width/height) will be matched
        (e.g. LCCS land cover, 10 m). This raster is NOT modified.
    resampling : rasterio.warp.Resampling
        Resampling.nearest   -> categorical data (e.g. land cover class codes)
        Resampling.bilinear  -> continuous data (e.g. % tree cover, NDVI)
    """
    with rasterio.open(reference_path) as ref:
        ref_transform = ref.transform
        ref_crs       = ref.crs
        ref_shape     = (ref.height, ref.width)

    with rasterio.open(source_path) as src:
        out_array = np.full(ref_shape, np.nan, dtype=np.float32)
        reproject(
            source=rasterio.band(src, 1),
            destination=out_array,
            src_transform=src.transform,
            src_crs=src.crs,
            src_nodata=src.nodata,
            dst_transform=ref_transform,
            dst_crs=ref_crs,
            dst_nodata=np.nan,
            resampling=resampling
        )
    print(f"✅ Aligned {os.path.basename(source_path)} → reference grid {ref_shape} "
          f"({resampling.name} resampling)")
    return out_array


# --- Load all rasters ---
# LCCS is our reference grid (10 m) — kept as-is, never resampled.
lccs_array, lccs_meta, lccs_bounds, lccs_crs = load_raster(LCCS_MAP_PATH, "LCCS L4 map")

# Every other raster (NDVI current, NDVI reference, Hansen tree cover) is at a
# different native resolution/extent than LCCS, so each one is reprojected and
# resampled onto the LCCS 10 m grid before any pixel-wise math is done. All are
# continuous variables, so bilinear interpolation is used throughout.
ndvi_current   = align_raster(NDVI_CURRENT_PATH,   LCCS_MAP_PATH, resampling=Resampling.bilinear)
ndvi_reference = align_raster(NDVI_REFERENCE_PATH, LCCS_MAP_PATH, resampling=Resampling.bilinear)
canopy_cover   = align_raster(CANOPY_COVER_PATH,   LCCS_MAP_PATH, resampling=Resampling.bilinear)

ndvi_cur_meta = lccs_meta.copy()
ndvi_ref_meta = lccs_meta.copy()
canopy_meta   = lccs_meta.copy()

# Sanity check — every layer must now share the LCCS grid shape
for name, arr in [("lccs_array", lccs_array), ("ndvi_current", ndvi_current),
                   ("ndvi_reference", ndvi_reference), ("canopy_cover", canopy_cover)]:
    assert arr.shape == lccs_array.shape, f"{name} shape {arr.shape} != LCCS shape {lccs_array.shape}"

print("ℹ️  Real file loading is commented out.")
print("   Running with SYNTHETIC demo data — replace paths above to use real files.")
print(f"✅ All layers aligned to common grid: {lccs_array.shape}")


In [ ]:
'''
# ============================================================
# SYNTHETIC DEMO DATA
# Replace this block with your real loaded rasters
# ============================================================

np.random.seed(42)
HEIGHT, WIDTH = 200, 300

# Simulate a realistic spatial pattern using Gaussian blobs
from scipy.ndimage import gaussian_filter

# Simulate LCCS L4 map (select a subset of codes for demo)
DEMO_CODES = [111, 112, 131, 211, 311, 312, 411, 511, 610, 710, 810]
raw_noise  = np.random.randint(0, len(DEMO_CODES), (HEIGHT, WIDTH))
smooth     = (gaussian_filter(raw_noise.astype(float), sigma=12)).astype(int)
smooth     = np.clip(smooth, 0, len(DEMO_CODES)-1)
lccs_array = np.vectorize(lambda x: DEMO_CODES[x])(smooth).astype(np.float32)

# Simulate NDVI layers (reference = slightly higher than current due to degradation)
ndvi_reference = np.clip(gaussian_filter(np.random.rand(HEIGHT, WIDTH), sigma=8) * 0.6 + 0.2, 0, 1).astype(np.float32)
degradation    = np.clip(gaussian_filter(np.random.rand(HEIGHT, WIDTH), sigma=5) * 0.3, 0, 0.5)
ndvi_current   = np.clip(ndvi_reference - degradation, 0, 1).astype(np.float32)

# Simulate canopy cover (%)
canopy_cover = np.clip(ndvi_reference * 80 + np.random.rand(HEIGHT, WIDTH) * 10, 0, 100).astype(np.float32)

# Fake metadata
from rasterio.transform import from_bounds as fb
lccs_meta = {
    'driver': 'GTiff', 'dtype': 'float32', 'nodata': -9999,
    'width': WIDTH, 'height': HEIGHT, 'count': 1,
    'crs': rasterio.crs.CRS.from_epsg(4326),
    'transform': fb(-10, -5, 10, 5, WIDTH, HEIGHT)
}

print(f"✅ Synthetic demo data created: {HEIGHT}×{WIDTH} pixels")
print(f"   LCCS classes present: {np.unique(lccs_array.astype(int)).tolist()}")
print(f"   NDVI reference range: [{ndvi_reference.min():.3f}, {ndvi_reference.max():.3f}]")
print(f"   NDVI current range:   [{ndvi_current.min():.3f}, {ndvi_current.max():.3f}]")
'''

In [ ]:
# ============================================================
# Visualise input layers
# ============================================================

# Build a colour map for LCCS classes
group_colors = {
    'forest':    '#1a7a1a',
    'shrubland': '#a0c060',
    'grassland': '#d4c860',
    'wetland':   '#5ba0c8',
    'cropland':  '#e8c878',
    'urban':     '#b0b0b0',
    'bare':      '#c8a878',
    'water':     '#2060c8',
    'ice':       '#e8f0ff',
}

unique_codes = np.unique(lccs_array[~np.isnan(lccs_array)]).astype(int)
lccs_rgb = np.zeros((*lccs_array.shape, 3), dtype=np.float32)
for code in unique_codes:
    group = code_to_group.get(code, 'urban')
    rgb   = mcolors.to_rgb(group_colors.get(group, '#808080'))
    mask_code = (lccs_array.astype(int) == code)
    for c, val in enumerate(rgb):
        lccs_rgb[:, :, c][mask_code] = val

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Input Layers — Ecosystem Condition Mapping', fontsize=14, fontweight='bold')

axes[0].imshow(lccs_rgb)
axes[0].set_title('LCCS L4 Land Cover (by ecosystem group)', fontsize=11)
axes[0].axis('off')
patches = [mpatches.Patch(color=v, label=k.capitalize()) for k, v in group_colors.items()]
axes[0].legend(handles=patches, loc='lower left', fontsize=7, ncol=2)

im1 = axes[1].imshow(ndvi_reference, cmap='YlGn', vmin=0, vmax=1)
axes[1].set_title('NDVI — Reference Period (Baseline)', fontsize=11)
axes[1].axis('off')
plt.colorbar(im1, ax=axes[1], shrink=0.7, label='NDVI')

im2 = axes[2].imshow(ndvi_current, cmap='YlGn', vmin=0, vmax=1)
axes[2].set_title('NDVI — Current Period', fontsize=11)
axes[2].axis('off')
plt.colorbar(im2, ax=axes[2], shrink=0.7, label='NDVI')

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, '01_input_layers.png'), dpi=150, bbox_inches='tight')
plt.show()
print("✅ Figure saved: 01_input_layers.png")

---
## 4. Compute Biophysical Condition Indicators

We calculate multiple indicators and then combine them into a composite condition score.

In [ ]:
# ============================================================
# Indicator 1: NDVI Deviation from Reference
# ============================================================

# Relative deviation: how much has NDVI dropped vs. reference?
# Positive value = degradation; Negative = improvement
with np.errstate(divide='ignore', invalid='ignore'):
    ndvi_deviation = np.where(
        ndvi_reference > 0,
        (ndvi_reference - ndvi_current) / ndvi_reference,
        np.nan
    ).astype(np.float32)

# Clip to [0, 1] — 0 = no degradation, 1 = complete loss
ndvi_deviation = np.clip(ndvi_deviation, 0, 1)

print("✅ Indicator 1: NDVI Deviation computed")
print(f"   Mean deviation: {np.nanmean(ndvi_deviation):.3f} ({np.nanmean(ndvi_deviation)*100:.1f}%)")
print(f"   Pixels with >25% deviation: {np.sum(ndvi_deviation > 0.25):,}")

In [ ]:
# ============================================================
# Indicator 2: Canopy Cover Change Index
# ============================================================
# ref_canopy_raster is built directly from lccs_array, so it is already on the
# 10 m LCCS grid. canopy_cover was resampled onto that same grid in the loading
# step above (bilinear), so the two arrays now have matching shapes.

assert canopy_cover.shape == lccs_array.shape, (
    f"canopy_cover {canopy_cover.shape} and lccs_array {lccs_array.shape} "
    "must share the same grid — check the resampling step above."
)

# Normalise canopy cover to [0, 1]; invert so higher = more degraded
# Expected reference canopy for forest = 70-100%, degraded < 30%

# Build per-pixel reference canopy based on LCCS class
REF_CANOPY = {
    'forest':    80.0,
    'shrubland': 40.0,
    'grassland': 5.0,
    'wetland':   30.0,
    'cropland':  20.0,
    'urban':     5.0,
    'bare':      0.0,
    'water':     0.0,
    'ice':       0.0,
}

# Build reference canopy raster from LCCS map (10 m grid)
ref_canopy_raster = np.full_like(lccs_array, np.nan)
for code in unique_codes:
    group = code_to_group.get(code, 'urban')
    ref_val = REF_CANOPY.get(group, 0.0)
    ref_canopy_raster[lccs_array.astype(int) == code] = ref_val

# Canopy deficit: how far below expected reference?
with np.errstate(divide='ignore', invalid='ignore'):
    canopy_deviation = np.where(
        ref_canopy_raster > 0,
        np.clip((ref_canopy_raster - canopy_cover) / ref_canopy_raster, 0, 1),
        0.0
    ).astype(np.float32)

print("✅ Indicator 2: Canopy Cover Deviation computed")
print(f"   Grid shape: {canopy_deviation.shape} (10 m, matches LCCS reference)")
print(f"   Mean canopy deficit: {np.nanmean(canopy_deviation):.3f}")


In [ ]:
# ============================================================
# Indicator 3: Fragmentation Index (Edge Density Proxy)
# ============================================================

# Detect edges between different LCCS classes using a Laplacian filter
# High edge density = high fragmentation = worse condition for natural ecosystems

lccs_int = np.nan_to_num(lccs_array, nan=0).astype(np.int32)

# Laplacian edge detection
laplacian_kernel = np.array([[0,-1,0],[-1,4,-1],[0,-1,0]])
edge_raw = np.abs(ndimage.convolve(lccs_int.astype(float), laplacian_kernel))

# Focal mean over 5x5 window to create neighbourhood fragmentation
frag_index = ndimage.uniform_filter(edge_raw, size=5)

# Normalise to [0, 1]
frag_max = np.nanpercentile(frag_index, 99)  # Use 99th percentile to avoid outlier influence
frag_index_norm = np.clip(frag_index / frag_max, 0, 1).astype(np.float32)

print("✅ Indicator 3: Fragmentation Index computed")
print(f"   Mean fragmentation: {np.nanmean(frag_index_norm):.3f}")

In [ ]:
# ============================================================
# Indicator 4: Human Pressure Index
# (Proximity to cropland & urban — as a proxy for anthropogenic pressure)
# ============================================================

# Identify cropland and urban pixels
pressure_classes = ['cropland', 'urban']
pressure_mask    = np.zeros_like(lccs_array, dtype=bool)
for code in unique_codes:
    if code_to_group.get(code, '') in pressure_classes:
        pressure_mask[lccs_array.astype(int) == code] = True

# Euclidean distance transform (pixels away from pressure source)
dist_to_pressure = ndimage.distance_transform_edt(~pressure_mask).astype(np.float32)

# Invert & normalise: close to pressure = high pressure index
max_dist = np.nanpercentile(dist_to_pressure, 99)
pressure_index = np.clip(1 - dist_to_pressure / max_dist, 0, 1).astype(np.float32)

# Zero out pressure index for cropland and urban pixels themselves
pressure_index[pressure_mask] = 0.0

print("✅ Indicator 4: Human Pressure Index computed")
print(f"   % pixels under high pressure (>0.5): {np.mean(pressure_index > 0.5)*100:.1f}%")

In [ ]:
# ============================================================
# Visualise all indicators
# ============================================================

indicators = {
    'NDVI Deviation\n(0=intact, 1=degraded)':        ndvi_deviation,
    'Canopy Cover Deficit\n(0=intact, 1=degraded)':  canopy_deviation,
    'Fragmentation Index\n(0=intact, 1=fragmented)': frag_index_norm,
    'Human Pressure Index\n(0=remote, 1=high)':      pressure_index,
}

fig, axes = plt.subplots(1, 4, figsize=(22, 5))
fig.suptitle('Biophysical Condition Indicators', fontsize=14, fontweight='bold')

for ax, (title, data) in zip(axes, indicators.items()):
    im = ax.imshow(data, cmap='RdYlGn_r', vmin=0, vmax=1)
    ax.set_title(title, fontsize=10)
    ax.axis('off')
    plt.colorbar(im, ax=ax, shrink=0.6)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, '02_condition_indicators.png'), dpi=150, bbox_inches='tight')
plt.show()
print("✅ Figure saved: 02_condition_indicators.png")

---
## 5. Composite Condition Score & SEEA EA Classification

In [ ]:
# ============================================================
# Weighted Composite Condition Degradation Score
# Weights reflect indicator importance (adjust as needed)
# ============================================================

INDICATOR_WEIGHTS = {
    'ndvi_deviation':   0.40,   # Vegetation productivity change
    'canopy_deviation': 0.25,   # Structural integrity
    'fragmentation':    0.20,   # Landscape connectivity
    'pressure':         0.15,   # Anthropogenic threat
}

# Composite degradation score [0=intact, 1=fully degraded]
composite_degradation = (
    ndvi_deviation    * INDICATOR_WEIGHTS['ndvi_deviation']   +
    canopy_deviation  * INDICATOR_WEIGHTS['canopy_deviation']  +
    frag_index_norm   * INDICATOR_WEIGHTS['fragmentation']     +
    pressure_index    * INDICATOR_WEIGHTS['pressure']
).astype(np.float32)

print("✅ Composite degradation score computed")
print(f"   Weights: {INDICATOR_WEIGHTS}")
print(f"   Score range: [{composite_degradation.min():.3f}, {composite_degradation.max():.3f}]")
print(f"   Mean score: {composite_degradation.mean():.3f}")

In [ ]:
# ============================================================
# SEEA EA 5-Point Condition Classification
# 5=Very Good, 4=Good, 3=Fair, 2=Poor, 1=Very Poor/Degraded
# ============================================================

def classify_condition(degradation_score, thresholds):
    """
    Classify degradation score into SEEA EA condition classes.
    Returns integer array: 5=Very Good ... 1=Very Poor
    """
    condition = np.full_like(degradation_score, np.nan, dtype=np.float32)
    condition[degradation_score <= thresholds['very_good']] = 5  # Very Good
    condition[(degradation_score > thresholds['very_good']) &
              (degradation_score <= thresholds['good'])]    = 4  # Good
    condition[(degradation_score > thresholds['good'])      &
              (degradation_score <= thresholds['fair'])]    = 3  # Fair
    condition[(degradation_score > thresholds['fair'])      &
              (degradation_score <= thresholds['poor'])]    = 2  # Poor
    condition[degradation_score > thresholds['poor']]       = 1  # Very Poor
    return condition


condition_map = classify_condition(composite_degradation, CONDITION_THRESHOLDS)

# Exclude non-natural ecosystems from condition assessment
NON_NATURAL_GROUPS = ['cropland', 'urban', 'water', 'ice']
non_natural_mask = np.zeros_like(lccs_array, dtype=bool)
for code in unique_codes:
    if code_to_group.get(code, '') in NON_NATURAL_GROUPS:
        non_natural_mask[lccs_array.astype(int) == code] = True

condition_map[non_natural_mask] = np.nan  # Mask out non-natural ecosystems

# Summary statistics
valid    = condition_map[~np.isnan(condition_map)]
total_px = valid.size

print("✅ SEEA EA Condition Classification complete")
print("\n   Condition class distribution (natural ecosystems only):")
labels = {5: 'Very Good', 4: 'Good', 3: 'Fair', 2: 'Poor', 1: 'Very Poor'}
for cls in [5, 4, 3, 2, 1]:
    count = np.sum(valid == cls)
    pct   = count / total_px * 100
    bar   = '█' * int(pct / 2)
    print(f"   {cls} – {labels[cls]:10s}: {count:6,} px ({pct:5.1f}%) {bar}")

---
## 6. Produce the Ecosystem Condition Map

In [ ]:
# ============================================================
# Main Ecosystem Condition Map
# ============================================================

CONDITION_COLORS = {
    5: '#1a7a1a',   # Very Good  — dark green
    4: '#78c850',   # Good       — light green
    3: '#f0d040',   # Fair       — yellow
    2: '#e87820',   # Poor       — orange
    1: '#c82020',   # Very Poor  — red
}
CONDITION_LABELS = {
    5: 'Very Good (score ≤5%)',
    4: 'Good (5–25%)',
    3: 'Fair (25–50%)',
    2: 'Poor (50–75%)',
    1: 'Very Poor / Degraded (>75%)',
}

# Build RGBA condition image
cond_rgb = np.zeros((*condition_map.shape, 4), dtype=np.float32)
for cls, hex_color in CONDITION_COLORS.items():
    rgba = (*mcolors.to_rgb(hex_color), 1.0)
    mask_cls = (condition_map == cls)
    for c, val in enumerate(rgba):
        cond_rgb[:, :, c][mask_cls] = val

# Non-natural classes in light grey
for c, val in enumerate((0.85, 0.85, 0.85, 0.7)):
    cond_rgb[:, :, c][non_natural_mask] = val

# NaN pixels transparent
nan_mask = np.isnan(condition_map) & ~non_natural_mask
cond_rgb[:, :, 3][nan_mask] = 0.0

# --- Plot ---
fig, axes = plt.subplots(1, 2, figsize=(16, 7),
                          gridspec_kw={'width_ratios': [2.5, 1]})
fig.patch.set_facecolor('#1c2833')

# Map panel
ax_map = axes[0]
ax_map.set_facecolor('#1c2833')
ax_map.imshow(cond_rgb, interpolation='bilinear')
ax_map.set_title('Ecosystem Condition Map\n(SEEA EA Classification)', 
                  color='white', fontsize=14, fontweight='bold', pad=12)
ax_map.axis('off')

# Legend
ax_leg = axes[1]
ax_leg.set_facecolor('#1c2833')
ax_leg.axis('off')
ax_leg.set_title('LEGEND', color='white', fontsize=12, fontweight='bold', loc='left')

legend_elements = [
    mpatches.Patch(facecolor=CONDITION_COLORS[cls], label=CONDITION_LABELS[cls])
    for cls in [5, 4, 3, 2, 1]
]
legend_elements.append(mpatches.Patch(facecolor='#d9d9d9', alpha=0.7, label='Non-natural (excluded)'))

leg = ax_leg.legend(
    handles=legend_elements,
    loc='center left', frameon=True, framealpha=0.15,
    facecolor='#2c3e50', edgecolor='white',
    labelcolor='white', fontsize=10,
    title='Condition Class', title_fontsize=11
)
leg.get_title().set_color('white')

# Add summary text
summary_text = "Indicators:\n"
for name, w in INDICATOR_WEIGHTS.items():
    summary_text += f"  • {name.replace('_',' ').title()} ({int(w*100)}%)\n"
summary_text += f"\nFramework: UN SEEA EA\nCRS: WGS84 (EPSG:4326)"
ax_leg.text(0.05, 0.15, summary_text, transform=ax_leg.transAxes,
            color='#aaaaaa', fontsize=9, va='bottom',
            bbox=dict(boxstyle='round', facecolor='#2c3e50', alpha=0.5, edgecolor='none'))

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, '03_ecosystem_condition_map.png'),
            dpi=200, bbox_inches='tight', facecolor=fig.get_facecolor())
plt.show()
print("✅ Main map saved: 03_ecosystem_condition_map.png")

---
## 7. Per-Ecosystem Type Statistics

In [ ]:
# ============================================================
# Condition breakdown per LCCS ecosystem group
# ============================================================

NATURAL_GROUPS = ['forest', 'shrubland', 'grassland', 'wetland', 'bare']

results = []
for code in unique_codes:
    group = code_to_group.get(code, 'unknown')
    if group not in NATURAL_GROUPS:
        continue
    
    px_mask     = (lccs_array.astype(int) == code) & (~np.isnan(condition_map))
    n_pixels    = np.sum(px_mask)
    if n_pixels == 0:
        continue
    
    cond_vals   = condition_map[px_mask]
    deg_vals    = composite_degradation[px_mask]
    mean_cond   = np.nanmean(cond_vals)
    
    for cls in [5, 4, 3, 2, 1]:
        count_cls = np.sum(cond_vals == cls)
        pct_cls   = count_cls / n_pixels * 100
        results.append({
            'lccs_code':        code,
            'ecosystem_type':   code_to_type.get(code, 'Unknown'),
            'eco_group':        group,
            'iucn_get':         code_to_biome.get(code, ''),
            'n_pixels':         n_pixels,
            'condition_class':  cls,
            'condition_label':  labels[cls],
            'pixel_count':      count_cls,
            'pct_area':         round(pct_cls, 2),
            'mean_condition':   round(mean_cond, 2),
            'mean_degradation': round(np.nanmean(deg_vals), 3),
        })

stats_df = pd.DataFrame(results)

# Summary pivot — % area per condition class, per ecosystem type
pivot = stats_df.pivot_table(
    index=['eco_group', 'ecosystem_type'],
    columns='condition_label',
    values='pct_area',
    aggfunc='sum',
    fill_value=0
).round(1)

# Ensure column order
ordered_cols = ['Very Good (score ≤5%)', 'Good (5–25%)', 'Fair (25–50%)',
                'Poor (50–75%)', 'Very Poor / Degraded (>75%)']
pivot = pivot.reindex(columns=[c for c in ordered_cols if c in pivot.columns])

print("✅ Per-ecosystem condition statistics:")
print()
print(pivot.to_string())
stats_df.to_csv(os.path.join(OUTPUT_DIR, 'condition_statistics.csv'), index=False)
print("\n✅ Full statistics saved: condition_statistics.csv")

In [ ]:
# ============================================================
# Stacked bar chart: condition by ecosystem group
# ============================================================

group_summary = stats_df.groupby(['eco_group', 'condition_label'])['pct_area'].sum().reset_index()

groups       = sorted(group_summary.eco_group.unique())
cond_classes = ['Very Good (score ≤5%)', 'Good (5–25%)', 'Fair (25–50%)',
                'Poor (50–75%)', 'Very Poor / Degraded (>75%)']
bar_colors   = ['#1a7a1a', '#78c850', '#f0d040', '#e87820', '#c82020']

fig, ax = plt.subplots(figsize=(12, 5))
fig.patch.set_facecolor('white')

bottom = np.zeros(len(groups))
for cls, color in zip(cond_classes, bar_colors):
    vals = []
    for g in groups:
        row = group_summary[(group_summary.eco_group == g) &
                            (group_summary.condition_label == cls)]
        vals.append(row.pct_area.sum() if len(row) else 0)
    vals = np.array(vals)
    ax.bar(groups, vals, bottom=bottom, color=color,
           label=cls.split('(')[0].strip(), edgecolor='white', linewidth=0.5)
    bottom += vals

ax.set_xlabel('Ecosystem Group', fontsize=11)
ax.set_ylabel('% of Ecosystem Area', fontsize=11)
ax.set_title('Ecosystem Condition by Ecosystem Group (SEEA EA)', fontsize=13, fontweight='bold')
ax.set_ylim(0, 115)
ax.legend(loc='upper right', fontsize=9, title='Condition Class', framealpha=0.9)
ax.spines[['top', 'right']].set_visible(False)
ax.set_xticklabels([g.capitalize() for g in groups], fontsize=10)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, '04_condition_by_ecosystem_group.png'), dpi=150, bbox_inches='tight')
plt.show()
print("✅ Chart saved: 04_condition_by_ecosystem_group.png")

---
## 8. Export Final Condition Raster (GeoTIFF)

In [ ]:
# ============================================================
# Export Condition Map as GeoTIFF
# ============================================================

output_tif = os.path.join(OUTPUT_DIR, 'ecosystem_condition_map.tif')

out_meta = lccs_meta.copy()
out_meta.update({
    'dtype':   'float32',
    'nodata':  -9999,
    'count':   1,
    'compress': 'lzw',
})

# Fill NaN with nodata value
condition_export = condition_map.copy()
condition_export[np.isnan(condition_export)] = -9999

with rasterio.open(output_tif, 'w', **out_meta) as dst:
    dst.write(condition_export, 1)
    # Write band description
    dst.update_tags(1,
        description     = 'Ecosystem Condition Score (SEEA EA)',
        values          = '5=Very Good, 4=Good, 3=Fair, 2=Poor, 1=Very Poor/Degraded, -9999=NoData',
        framework       = 'UN SEEA EA 2021',
        lccs_input      = 'FAO LCCS Level 4',
        indicators      = 'NDVI deviation, Canopy cover deficit, Fragmentation, Human pressure'
    )

print(f"✅ Condition raster exported: {output_tif}")
print(f"   Shape: {condition_export.shape}")
print(f"   Unique values: {np.unique(condition_export[condition_export != -9999]).tolist()}")

---
## 9. IUCN Red List of Ecosystems (RLE) — Collapse Risk Assessment

Based on the condition map, we can flag ecosystems at risk of collapse using criterion D (Degradation of abiotic/biotic processes).

In [ ]:
# ============================================================
# RLE Criterion D: Quantitative Analysis of Degradation
# An ecosystem is considered:
#   Collapsed (CO)     if >80% area in class 1 (Very Poor)
#   Critically Endan.  if >50% area in class 1–2
#   Endangered         if >30% area in class 1–2
#   Vulnerable         if >10% area in class 1–2
#   Least Concern      otherwise
# ============================================================

def rle_criterion_d(ecosystem_name, condition_values):
    n = len(condition_values)
    if n == 0:
        return 'Not Assessed'
    pct_collapsed  = np.sum(condition_values == 1) / n * 100
    pct_poor_below = np.sum(condition_values <= 2) / n * 100
    
    if pct_collapsed >= 80:
        return 'Collapsed (CO)'
    elif pct_poor_below >= 50:
        return 'Critically Endangered (CR)'
    elif pct_poor_below >= 30:
        return 'Endangered (EN)'
    elif pct_poor_below >= 10:
        return 'Vulnerable (VU)'
    else:
        return 'Least Concern (LC)'


rle_results = []
for code in unique_codes:
    group = code_to_group.get(code, 'unknown')
    if group not in NATURAL_GROUPS:
        continue
    px_mask   = (lccs_array.astype(int) == code) & (~np.isnan(condition_map))
    cond_vals = condition_map[px_mask]
    if len(cond_vals) == 0:
        continue
    rle_results.append({
        'lccs_code':       code,
        'ecosystem_type':  code_to_type.get(code, 'Unknown'),
        'iucn_get':        code_to_biome.get(code, ''),
        'n_pixels':        len(cond_vals),
        'mean_condition':  round(np.mean(cond_vals), 2),
        'pct_poor_degraded': round(np.sum(cond_vals <= 2) / len(cond_vals) * 100, 1),
        'rle_status':      rle_criterion_d(code, cond_vals)
    })

rle_df = pd.DataFrame(rle_results).sort_values('pct_poor_degraded', ascending=False)

# Colour-code the output
rle_color_map = {
    'Collapsed (CO)':           '\033[91m',  # red
    'Critically Endangered (CR)':'\033[91m',
    'Endangered (EN)':          '\033[93m',  # yellow
    'Vulnerable (VU)':          '\033[93m',
    'Least Concern (LC)':       '\033[92m',  # green
}
RESET = '\033[0m'

print("✅ IUCN RLE Criterion D Assessment:")
print(f"\n{'Ecosystem Type':<40} {'IUCN GET':<8} {'Mean Cond.':<12} {'% Poor/Deg':<12} RLE Status")
print("-" * 90)
for _, row in rle_df.iterrows():
    color = rle_color_map.get(row.rle_status, '')
    print(f"{row.ecosystem_type:<40} {row.iucn_get:<8} {row.mean_condition:<12} {row.pct_poor_degraded:<12} {color}{row.rle_status}{RESET}")

rle_df.to_csv(os.path.join(OUTPUT_DIR, 'rle_assessment.csv'), index=False)
print("\n✅ RLE results saved: rle_assessment.csv")

---
## 10. Summary Dashboard

In [ ]:
# ============================================================
# Summary 4-panel dashboard
# ============================================================

fig = plt.figure(figsize=(18, 12))
fig.patch.set_facecolor('#f8f9fa')
fig.suptitle('Ecosystem Condition Mapping — Summary Dashboard',
             fontsize=16, fontweight='bold', y=0.98)

gs = fig.add_gridspec(2, 3, hspace=0.35, wspace=0.3)

# Panel 1: Condition Map
ax1 = fig.add_subplot(gs[0, :2])
ax1.imshow(cond_rgb, interpolation='bilinear')
ax1.set_title('Ecosystem Condition Map (SEEA EA)', fontsize=12, fontweight='bold')
ax1.axis('off')
patches = [mpatches.Patch(color=CONDITION_COLORS[c], label=labels[c]) for c in [5,4,3,2,1]]
patches.append(mpatches.Patch(color='#d9d9d9', alpha=0.7, label='Non-natural'))
ax1.legend(handles=patches, loc='lower left', fontsize=8, framealpha=0.85)

# Panel 2: Pie chart — overall condition
ax2 = fig.add_subplot(gs[0, 2])
all_cond = condition_map[~np.isnan(condition_map)]
pie_sizes  = [np.sum(all_cond == c) for c in [5,4,3,2,1]]
pie_colors = [CONDITION_COLORS[c] for c in [5,4,3,2,1]]
pie_labels = [f"{labels[c]}\n{s/sum(pie_sizes)*100:.1f}%" for c, s in zip([5,4,3,2,1], pie_sizes)]
ax2.pie(pie_sizes, labels=pie_labels, colors=pie_colors,
        startangle=90, textprops={'fontsize': 8})
ax2.set_title('Overall Condition\n(natural ecosystems)', fontsize=11, fontweight='bold')

# Panel 3: Degradation score histogram
ax3 = fig.add_subplot(gs[1, 0])
valid_deg = composite_degradation[~np.isnan(condition_map)]
ax3.hist(valid_deg, bins=50, color='steelblue', edgecolor='white', linewidth=0.3, alpha=0.85)
for thr, lbl, col in [
    (CONDITION_THRESHOLDS['very_good'], 'Very Good', CONDITION_COLORS[5]),
    (CONDITION_THRESHOLDS['good'],      'Good',       CONDITION_COLORS[4]),
    (CONDITION_THRESHOLDS['fair'],      'Fair',       CONDITION_COLORS[3]),
    (CONDITION_THRESHOLDS['poor'],      'Poor',       CONDITION_COLORS[2]),
]:
    ax3.axvline(thr, color=col, linewidth=1.5, linestyle='--', label=lbl)
ax3.set_xlabel('Composite Degradation Score', fontsize=10)
ax3.set_ylabel('Pixel Count', fontsize=10)
ax3.set_title('Degradation Score Distribution', fontsize=11, fontweight='bold')
ax3.legend(fontsize=7)
ax3.spines[['top','right']].set_visible(False)

# Panel 4: Indicator correlation matrix
ax4 = fig.add_subplot(gs[1, 1])
indicator_arrays = np.stack([
    ndvi_deviation.flatten(),
    canopy_deviation.flatten(),
    frag_index_norm.flatten(),
    pressure_index.flatten(),
], axis=1)
corr_matrix = np.corrcoef(indicator_arrays.T)
im = ax4.imshow(corr_matrix, cmap='RdBu_r', vmin=-1, vmax=1)
ind_names = ['NDVI\nDeviation', 'Canopy\nDeficit', 'Fragm.\nIndex', 'Human\nPressure']
ax4.set_xticks(range(4)); ax4.set_yticks(range(4))
ax4.set_xticklabels(ind_names, fontsize=8)
ax4.set_yticklabels(ind_names, fontsize=8)
for i in range(4):
    for j in range(4):
        ax4.text(j, i, f'{corr_matrix[i,j]:.2f}', ha='center', va='center',
                 fontsize=9, color='white' if abs(corr_matrix[i,j]) > 0.5 else 'black')
plt.colorbar(im, ax=ax4, shrink=0.7)
ax4.set_title('Indicator Correlation Matrix', fontsize=11, fontweight='bold')

# Panel 5: RLE assessment bar
ax5 = fig.add_subplot(gs[1, 2])
rle_order   = ['Collapsed (CO)', 'Critically Endangered (CR)', 'Endangered (EN)',
                'Vulnerable (VU)', 'Least Concern (LC)']
rle_counts  = rle_df.rle_status.value_counts().reindex(rle_order, fill_value=0)
rle_bar_colors = ['#8b0000', '#c82020', '#e87820', '#f0d040', '#1a7a1a']
bars = ax5.barh(rle_counts.index, rle_counts.values,
                color=rle_bar_colors[:len(rle_counts)], edgecolor='white')
ax5.set_xlabel('Number of Ecosystem Types', fontsize=10)
ax5.set_title('IUCN RLE Status\n(Criterion D)', fontsize=11, fontweight='bold')
ax5.spines[['top','right']].set_visible(False)
for bar in bars:
    width = bar.get_width()
    if width > 0:
        ax5.text(width + 0.05, bar.get_y() + bar.get_height()/2,
                 f'{int(width)}', va='center', fontsize=9)

plt.savefig(os.path.join(OUTPUT_DIR, '05_summary_dashboard.png'), dpi=200, bbox_inches='tight',
            facecolor=fig.get_facecolor())
plt.show()
print("✅ Dashboard saved: 05_summary_dashboard.png")

---
## 11. Outputs Summary

In [ ]:
print("=" * 60)
print("  ECOSYSTEM CONDITION MAPPING — OUTPUTS")
print("=" * 60)
print()
outputs = [
    ("ecosystem_condition_map.tif",         "GeoTIFF — SEEA EA condition raster (1–5)"),
    ("condition_statistics.csv",            "CSV — Per-ecosystem condition breakdown"),
    ("rle_assessment.csv",                  "CSV — IUCN RLE Criterion D assessment"),
    ("01_input_layers.png",                 "Figure — LCCS map + NDVI layers"),
    ("02_condition_indicators.png",         "Figure — 4 biophysical indicators"),
    ("03_ecosystem_condition_map.png",      "Figure — Main condition map"),
    ("04_condition_by_ecosystem_group.png", "Figure — Stacked bar by ecosystem group"),
    ("05_summary_dashboard.png",            "Figure — Full summary dashboard"),
]
for fname, desc in outputs:
    path = os.path.join(OUTPUT_DIR, fname)
    exists = '✅' if os.path.exists(path) else '❌'
    print(f"  {exists}  {fname:<42} {desc}")
print()
print(f"  Output directory: {os.path.abspath(OUTPUT_DIR)}")
print("=" * 60)

---
## Appendix B — Customisation Guide

| Task | Where to modify |
|---|---|
| Add/modify LCCS L4 classes | `crosswalk_data` list in Section 2 |
| Change indicator weights | `INDICATOR_WEIGHTS` dict in Section 5 |
| Modify condition thresholds | `CONDITION_THRESHOLDS` in Section 1 |
| Add new indicators | Add computation in Section 4, include in composite score |
| Use real rasters | Uncomment `load_raster()` calls in Section 3 |
| Run per-polygon statistics | Use `rasterstats.zonal_stats()` with your ecosystem polygons |
| Scale to national/global | Use the GEE script in Appendix A |

### Recommended Additional Indicators
- **Fire disturbance**: MODIS FIRMS burned area → binary or severity raster  
- **Soil degradation**: BSI (Bare Soil Index) = `(SWIR + Red) / (NIR + Blue)`  
- **Water stress**: NDWI, JRC global surface water permanence  
- **Invasive species**: spectral anomaly or species distribution models  
- **Connectivity**: graph-based landscape connectivity (e.g. `grainscape` in R)  